[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_08/notebook_8_5_rag.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 8.5: Retrieval-Augmented Generation (RAG) for Clinical Decision Support

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand the RAG architecture and why it's essential for clinical applications
2. Implement document chunking strategies for medical literature
3. Build semantic search using embeddings and vector databases
4. Create a production-ready RAG pipeline for clinical guidelines
5. Evaluate retrieval quality and implement ranking improvements
6. Handle citation tracking and source attribution

## Clinical Context

**The Challenge**: LLMs have knowledge cutoff dates and may hallucinate medical information. They cannot access:
- Institution-specific clinical guidelines
- Recent research (post-training cutoff)
- Patient-specific medical records
- Local formulary and treatment protocols

**The Solution**: RAG combines retrieval (finding relevant documents) with generation (LLM response) to ground answers in authoritative sources.

**Clinical Use Cases**:
- Guidelines-based clinical decision support
- Literature review for differential diagnosis
- Drug interaction checking with latest safety alerts
- Patient education with verified medical information

**Real-World Impact**:
- Mayo Clinic uses RAG for evidence-based treatment recommendations
- UK NHS implements RAG for NICE guideline queries
- Research shows 40-60% reduction in hallucinations with RAG vs. vanilla LLMs

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional
import re
import json
from dataclasses import dataclass
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# For embeddings (simulated in this notebook)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")
print("\nNote: This notebook demonstrates RAG concepts with simulated embeddings.")
print("For production, use: OpenAI embeddings, Sentence-Transformers, or Cohere.")

## Part 1: Understanding RAG Architecture

### RAG vs. Standard LLM

**Standard LLM**:
```
User Query → LLM → Response (based on training data only)
```

**RAG Pipeline**:
```
User Query → Retrieval System → Relevant Documents → LLM (with context) → Response + Citations
```

### RAG Components

1. **Document Corpus**: Clinical guidelines, research papers, protocols
2. **Chunking Strategy**: Split documents into retrievable units
3. **Embedding Model**: Convert text to dense vectors
4. **Vector Database**: Store and search embeddings efficiently
5. **Retrieval**: Find k most relevant chunks for query
6. **Prompt Construction**: Combine query + retrieved context
7. **LLM Generation**: Generate response grounded in retrieved documents
8. **Citation Tracking**: Attribute claims to sources

In [ ]:
@dataclass
class Document:
    """Represents a clinical document in our corpus."""
    doc_id: str
    title: str
    content: str
    metadata: Dict

@dataclass
class Chunk:
    """Represents a chunk of a document."""
    chunk_id: str
    doc_id: str
    content: str
    start_idx: int
    end_idx: int
    metadata: Dict
    embedding: Optional[np.ndarray] = None

@dataclass
class RetrievalResult:
    """Represents a retrieved chunk with relevance score."""
    chunk: Chunk
    score: float
    rank: int

print("RAG data structures defined!")

## Part 2: Creating a Clinical Document Corpus

For demonstration, we'll use synthetic clinical guidelines. In production, you would:
- Import NICE guidelines, UpToDate articles, local protocols
- Process PDFs, HTML, or structured databases
- Maintain version control and update schedules

In [ ]:
# Synthetic clinical guideline corpus
clinical_documents = [
    Document(
        doc_id="GUIDE_001",
        title="Acute Myocardial Infarction Management Guidelines",
        content="""
        SECTION 1: INITIAL ASSESSMENT
        Patients presenting with chest pain should undergo immediate ECG within 10 minutes of arrival.
        ST-elevation myocardial infarction (STEMI) is diagnosed when ST elevation >1mm in two contiguous leads.

        SECTION 2: IMMEDIATE MANAGEMENT
        STEMI patients require immediate reperfusion therapy. Primary PCI is preferred if available within 120 minutes.
        Dual antiplatelet therapy (aspirin 300mg + ticagrelor 180mg) should be administered immediately.

        SECTION 3: CONTRAINDICATIONS
        Thrombolysis is contraindicated in patients with: active bleeding, recent major surgery (within 3 weeks),
        history of hemorrhagic stroke, or blood pressure >180/110 mmHg despite treatment.

        SECTION 4: POST-INTERVENTION CARE
        All patients should receive beta-blockers within 24 hours unless contraindicated. ACE inhibitors should be
        started within 24-48 hours for patients with LV dysfunction or heart failure.
        """,
        metadata={"source": "AHA/ACC Guidelines 2023", "category": "cardiology", "evidence_level": "A"}
    ),
    Document(
        doc_id="GUIDE_002",
        title="Type 2 Diabetes Management Protocol",
        content="""
        SECTION 1: DIAGNOSIS
        Type 2 diabetes is diagnosed when: fasting glucose ≥126 mg/dL on two occasions, or HbA1c ≥6.5%,
        or random glucose ≥200 mg/dL with symptoms.

        SECTION 2: FIRST-LINE TREATMENT
        Metformin is the first-line pharmacological therapy for type 2 diabetes, starting at 500mg daily with meals.
        Titrate by 500mg weekly to target dose of 2000mg daily (divided doses). Monitor renal function quarterly.

        SECTION 3: SECOND-LINE AGENTS
        If HbA1c remains >7% after 3 months on metformin, add: GLP-1 agonist (for weight loss benefit),
        SGLT2 inhibitor (for cardiovascular benefit), or DPP-4 inhibitor (if GI side effects with other agents).

        SECTION 4: MONITORING
        Check HbA1c every 3 months until stable, then every 6 months. Annual screening for: retinopathy,
        nephropathy (urine albumin-to-creatinine ratio), neuropathy (monofilament test), and foot examination.
        """,
        metadata={"source": "ADA Standards of Care 2024", "category": "endocrinology", "evidence_level": "A"}
    ),
    Document(
        doc_id="GUIDE_003",
        title="Community-Acquired Pneumonia Treatment Guidelines",
        content="""
        SECTION 1: SEVERITY ASSESSMENT
        Use CURB-65 score to assess severity: Confusion, Urea >7 mmol/L, Respiratory rate ≥30, Blood pressure <90/60,
        age ≥65. Score 0-1: outpatient treatment; 2: consider admission; 3-5: severe pneumonia requiring admission.

        SECTION 2: OUTPATIENT TREATMENT
        Previously healthy patients: amoxicillin 1g three times daily for 5-7 days. Alternative: doxycycline 100mg
        twice daily. Patients with comorbidities: amoxicillin-clavulanate 875/125mg twice daily plus azithromycin.

        SECTION 3: INPATIENT TREATMENT
        Non-severe: amoxicillin-clavulanate 1.2g IV three times daily plus clarithromycin 500mg IV twice daily.
        Severe pneumonia: piperacillin-tazobactam 4.5g IV every 6 hours plus clarithromycin 500mg IV twice daily.

        SECTION 4: MONITORING AND FOLLOW-UP
        Clinical improvement expected within 48-72 hours. If no improvement, consider alternative diagnosis or
        resistant organism. Chest X-ray follow-up at 6 weeks for patients >50 years or smokers.
        """,
        metadata={"source": "IDSA/ATS Guidelines 2023", "category": "infectious_disease", "evidence_level": "A"}
    ),
    Document(
        doc_id="GUIDE_004",
        title="Hypertension Management Guidelines",
        content="""
        SECTION 1: DIAGNOSIS
        Hypertension is diagnosed when: office BP ≥140/90 mmHg on two separate occasions, or ambulatory/home
        BP ≥135/85 mmHg. Stage 2: BP ≥160/100 mmHg. Recommend ABPM for confirmation.

        SECTION 2: LIFESTYLE MODIFICATIONS
        All patients should receive counseling on: sodium reduction (<2g/day), DASH diet, weight loss if BMI >25,
        aerobic exercise 150 min/week, alcohol limitation, and smoking cessation.

        SECTION 3: PHARMACOLOGICAL TREATMENT
        First-line agents: ACE inhibitor (e.g., lisinopril 10mg daily) or ARB, calcium channel blocker (amlodipine
        5mg daily), or thiazide diuretic (hydrochlorothiazide 12.5mg daily). For Black patients, prefer CCB or thiazide.

        SECTION 4: TREATMENT TARGETS
        Target BP: <140/90 mmHg for most patients, <130/80 mmHg for patients with diabetes or CKD. For patients
        >65 years, consider <150/90 mmHg if frail. Titrate therapy every 2-4 weeks until target achieved.
        """,
        metadata={"source": "ESC/ESH Guidelines 2023", "category": "cardiology", "evidence_level": "A"}
    ),
    Document(
        doc_id="GUIDE_005",
        title="Acute Kidney Injury Management Protocol",
        content="""
        SECTION 1: DEFINITION AND STAGING
        AKI is defined by KDIGO criteria: increase in serum creatinine ≥0.3 mg/dL within 48 hours, or increase
        ≥1.5x baseline within 7 days, or urine output <0.5 mL/kg/h for 6 hours. Stage 1: 1.5-1.9x baseline.

        SECTION 2: INITIAL EVALUATION
        Identify cause: pre-renal (volume depletion), intrinsic (acute tubular necrosis, glomerulonephritis),
        or post-renal (obstruction). Check: urinalysis, renal ultrasound, review medications.

        SECTION 3: MANAGEMENT
        Pre-renal AKI: fluid resuscitation with isotonic saline. Stop nephrotoxic medications (NSAIDs, ACE inhibitors,
        aminoglycosides). Adjust drug dosing for reduced GFR. Monitor fluid balance, electrolytes, and creatinine daily.

        SECTION 4: RENAL REPLACEMENT THERAPY
        Indications for dialysis: severe hyperkalemia (K >6.5 mmol/L), metabolic acidosis (pH <7.1), fluid overload
        causing pulmonary edema, uremic complications (pericarditis, encephalopathy), or BUN >100 mg/dL.
        """,
        metadata={"source": "KDIGO Guidelines 2023", "category": "nephrology", "evidence_level": "A"}
    )
]

print(f"Created corpus with {len(clinical_documents)} clinical guidelines")
print("\nDocuments:")
for doc in clinical_documents:
    print(f"  - {doc.doc_id}: {doc.title}")

## Part 3: Document Chunking Strategies

### Why Chunking Matters

- **LLM Context Limits**: Most LLMs have token limits (4K-128K). Can't pass entire corpus.
- **Retrieval Precision**: Smaller chunks allow more precise matching to queries.
- **Semantic Coherence**: Chunks should contain complete ideas.

### Chunking Strategies

1. **Fixed-Size**: Split every N tokens (simple but breaks sentences)
2. **Sentence-Based**: Split on sentence boundaries (better coherence)
3. **Paragraph-Based**: Split on paragraphs (preserves context)
4. **Semantic**: Use NLP to identify topic boundaries (most sophisticated)
5. **Sliding Window**: Overlapping chunks to avoid boundary issues

For clinical documents with sections, we'll use **section-based chunking** with overlap.

In [ ]:
class DocumentChunker:
    """
    Chunks clinical documents intelligently.
    Strategy: Section-based with sentence-level splitting for long sections.
    """

    def __init__(self, max_chunk_tokens: int = 512, overlap_tokens: int = 50):
        """
        Args:
            max_chunk_tokens: Maximum tokens per chunk (approximate)
            overlap_tokens: Number of tokens to overlap between chunks
        """
        self.max_chunk_tokens = max_chunk_tokens
        self.overlap_tokens = overlap_tokens

    def chunk_document(self, document: Document) -> List[Chunk]:
        """Chunk a document into retrievable units."""
        chunks = []

        # Split by sections (for our structured guidelines)
        sections = self._split_sections(document.content)

        chunk_idx = 0
        for section_title, section_content in sections:
            # Each section becomes a chunk (with title for context)
            chunk_content = f"{section_title}\n{section_content}"

            # Approximate token count (4 chars ≈ 1 token)
            estimated_tokens = len(chunk_content) // 4

            if estimated_tokens <= self.max_chunk_tokens:
                # Section fits in one chunk
                chunk = Chunk(
                    chunk_id=f"{document.doc_id}_chunk_{chunk_idx}",
                    doc_id=document.doc_id,
                    content=chunk_content.strip(),
                    start_idx=0,  # Simplified for this demo
                    end_idx=len(chunk_content),
                    metadata={
                        **document.metadata,
                        'section': section_title,
                        'doc_title': document.title
                    }
                )
                chunks.append(chunk)
                chunk_idx += 1
            else:
                # Section too long - split into sentences
                sentences = self._split_sentences(section_content)
                current_chunk = section_title + "\n"

                for sentence in sentences:
                    if len(current_chunk + sentence) // 4 <= self.max_chunk_tokens:
                        current_chunk += sentence + " "
                    else:
                        # Save current chunk
                        if current_chunk.strip():
                            chunk = Chunk(
                                chunk_id=f"{document.doc_id}_chunk_{chunk_idx}",
                                doc_id=document.doc_id,
                                content=current_chunk.strip(),
                                start_idx=0,
                                end_idx=len(current_chunk),
                                metadata={
                                    **document.metadata,
                                    'section': section_title,
                                    'doc_title': document.title
                                }
                            )
                            chunks.append(chunk)
                            chunk_idx += 1

                        # Start new chunk with overlap (last sentence)
                        current_chunk = section_title + "\n" + sentence + " "

                # Save final chunk
                if current_chunk.strip():
                    chunk = Chunk(
                        chunk_id=f"{document.doc_id}_chunk_{chunk_idx}",
                        doc_id=document.doc_id,
                        content=current_chunk.strip(),
                        start_idx=0,
                        end_idx=len(current_chunk),
                        metadata={
                            **document.metadata,
                            'section': section_title,
                            'doc_title': document.title
                        }
                    )
                    chunks.append(chunk)
                    chunk_idx += 1

        return chunks

    def _split_sections(self, content: str) -> List[Tuple[str, str]]:
        """Split document into sections based on 'SECTION N:' pattern."""
        sections = []
        lines = content.strip().split('\n')

        current_section = ""
        current_content = []

        for line in lines:
            if re.match(r'^\s*SECTION \d+:', line):
                # Save previous section
                if current_section:
                    sections.append((current_section, '\n'.join(current_content)))

                # Start new section
                current_section = line.strip()
                current_content = []
            else:
                current_content.append(line)

        # Save last section
        if current_section:
            sections.append((current_section, '\n'.join(current_content)))

        return sections

    def _split_sentences(self, text: str) -> List[str]:
        """Simple sentence splitter."""
        # Split on periods followed by space and capital letter
        sentences = re.split(r'(?<=\.)\s+(?=[A-Z])', text)
        return [s.strip() for s in sentences if s.strip()]

# Test chunking
chunker = DocumentChunker(max_chunk_tokens=512, overlap_tokens=50)

all_chunks = []
for doc in clinical_documents:
    chunks = chunker.chunk_document(doc)
    all_chunks.extend(chunks)
    print(f"\n{doc.title}:")
    print(f"  Generated {len(chunks)} chunks")
    for i, chunk in enumerate(chunks[:2]):  # Show first 2 chunks
        print(f"  Chunk {i}: {chunk.content[:100]}...")

print(f"\n\nTotal chunks in corpus: {len(all_chunks)}")

## Part 4: Embeddings and Vector Database

### Embedding Models for Healthcare

**Production Options**:
1. **OpenAI `text-embedding-3-small`**: General-purpose, 1536 dimensions, $0.02/1M tokens
2. **Sentence-Transformers `all-MiniLM-L6-v2`**: Free, local, 384 dimensions
3. **BioBERT/ClinicalBERT embeddings**: Domain-specific for medical text
4. **Cohere `embed-english-v3.0`**: Multilingual, compression support

**Vector Databases**:
- **Pinecone**: Managed, scalable, $70/month for 5M vectors
- **Weaviate**: Open-source, on-premise option
- **ChromaDB**: Lightweight, good for prototyping
- **FAISS**: Facebook AI, very fast for large-scale

For this notebook, we'll use TF-IDF as a simple embedding proxy.

In [ ]:
class SimpleVectorDatabase:
    """
    Simplified vector database using TF-IDF for demonstration.
    In production, use Pinecone, Weaviate, or ChromaDB with proper embeddings.
    """

    def __init__(self):
        self.chunks: List[Chunk] = []
        self.vectorizer = TfidfVectorizer(
            max_features=1000,
            stop_words='english',
            ngram_range=(1, 2)
        )
        self.chunk_vectors = None

    def add_chunks(self, chunks: List[Chunk]):
        """Add chunks to the database and compute embeddings."""
        self.chunks = chunks

        # Extract text for vectorization
        texts = [chunk.content for chunk in chunks]

        # Compute TF-IDF vectors (proxy for embeddings)
        self.chunk_vectors = self.vectorizer.fit_transform(texts)

        print(f"Added {len(chunks)} chunks to vector database")
        print(f"Vector dimensions: {self.chunk_vectors.shape[1]}")

    def search(self, query: str, top_k: int = 5) -> List[RetrievalResult]:
        """Search for most relevant chunks."""
        if self.chunk_vectors is None:
            raise ValueError("No chunks in database. Call add_chunks() first.")

        # Vectorize query
        query_vector = self.vectorizer.transform([query])

        # Compute cosine similarity
        similarities = cosine_similarity(query_vector, self.chunk_vectors)[0]

        # Get top-k indices
        top_indices = np.argsort(similarities)[::-1][:top_k]

        # Create retrieval results
        results = []
        for rank, idx in enumerate(top_indices):
            result = RetrievalResult(
                chunk=self.chunks[idx],
                score=float(similarities[idx]),
                rank=rank + 1
            )
            results.append(result)

        return results

    def get_statistics(self) -> Dict:
        """Get database statistics."""
        return {
            'total_chunks': len(self.chunks),
            'vector_dimensions': self.chunk_vectors.shape[1] if self.chunk_vectors is not None else 0,
            'unique_documents': len(set(chunk.doc_id for chunk in self.chunks))
        }

# Create and populate vector database
vector_db = SimpleVectorDatabase()
vector_db.add_chunks(all_chunks)

# Show statistics
stats = vector_db.get_statistics()
print("\nDatabase Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## Part 5: Retrieval and Ranking

### Testing Retrieval Quality

In [ ]:
def display_retrieval_results(query: str, results: List[RetrievalResult]):
    """Display retrieval results in a readable format."""
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")

    for result in results:
        print(f"\n[Rank {result.rank}] Score: {result.score:.3f}")
        print(f"Document: {result.chunk.metadata['doc_title']}")
        print(f"Section: {result.chunk.metadata.get('section', 'N/A')}")
        print(f"Content:\n{result.chunk.content[:300]}...")
        print("-" * 80)

# Test queries
test_queries = [
    "What is the first-line treatment for type 2 diabetes?",
    "When should I give thrombolysis for STEMI?",
    "What antibiotics for community-acquired pneumonia?",
    "How do I diagnose acute kidney injury?"
]

for query in test_queries[:2]:  # Show first 2 queries
    results = vector_db.search(query, top_k=3)
    display_retrieval_results(query, results)

## Part 6: RAG Pipeline - Putting It All Together

### Complete RAG System

In [ ]:
class ClinicalRAGSystem:
    """
    Complete RAG system for clinical decision support.
    """

    def __init__(self, vector_db: SimpleVectorDatabase):
        self.vector_db = vector_db
        self.retrieval_history = []

    def retrieve(self, query: str, top_k: int = 3) -> List[RetrievalResult]:
        """Retrieve relevant chunks for a query."""
        results = self.vector_db.search(query, top_k=top_k)

        # Log retrieval
        self.retrieval_history.append({
            'query': query,
            'num_results': len(results),
            'top_score': results[0].score if results else 0.0
        })

        return results

    def construct_prompt(self, query: str, results: List[RetrievalResult]) -> str:
        """
        Construct prompt with retrieved context for LLM.
        """
        # Build context from retrieved chunks
        context_parts = []
        for i, result in enumerate(results, 1):
            context_parts.append(
                f"[Source {i}: {result.chunk.metadata['doc_title']} - "
                f"{result.chunk.metadata.get('section', 'Section')}]\n"
                f"{result.chunk.content}\n"
            )

        context = "\n".join(context_parts)

        # Construct prompt with instructions
        prompt = f"""You are a clinical decision support assistant. Answer the question based ONLY on the provided clinical guidelines.

IMPORTANT INSTRUCTIONS:
1. Base your answer exclusively on the provided sources
2. Cite sources using [Source N] format
3. If information is not in the sources, state: "The provided guidelines do not contain information about..."
4. Include relevant dosages, contraindications, and monitoring requirements
5. Note the evidence level if provided

CLINICAL GUIDELINES:
{context}

QUESTION: {query}

ANSWER (with citations):"""

        return prompt

    def generate_response_simulated(self, query: str, top_k: int = 3) -> Dict:
        """
        Simulated RAG response (without calling actual LLM).
        In production, pass prompt to OpenAI/Anthropic/etc.
        """
        # Step 1: Retrieve relevant chunks
        results = self.retrieve(query, top_k=top_k)

        if not results:
            return {
                'query': query,
                'answer': "No relevant information found in clinical guidelines.",
                'sources': [],
                'confidence': 0.0
            }

        # Step 2: Construct prompt
        prompt = self.construct_prompt(query, results)

        # Step 3: Simulated LLM response (extract key info from top result)
        top_chunk = results[0].chunk

        # Simple extraction of relevant sentences
        sentences = top_chunk.content.split('.')
        relevant_sentences = [s.strip() for s in sentences if len(s.strip()) > 20][:3]

        simulated_answer = " ".join(relevant_sentences) + ". [Source 1]"

        # Step 4: Return structured response
        return {
            'query': query,
            'answer': simulated_answer,
            'sources': [
                {
                    'title': result.chunk.metadata['doc_title'],
                    'section': result.chunk.metadata.get('section', 'N/A'),
                    'relevance_score': result.score,
                    'evidence_level': result.chunk.metadata.get('evidence_level', 'N/A')
                }
                for result in results
            ],
            'confidence': results[0].score,
            'prompt': prompt  # For debugging
        }

    def get_retrieval_statistics(self) -> Dict:
        """Get statistics on retrieval performance."""
        if not self.retrieval_history:
            return {}

        scores = [r['top_score'] for r in self.retrieval_history]
        return {
            'total_queries': len(self.retrieval_history),
            'avg_top_score': np.mean(scores),
            'min_top_score': np.min(scores),
            'max_top_score': np.max(scores)
        }

# Initialize RAG system
rag_system = ClinicalRAGSystem(vector_db)
print("RAG system initialized!")

### Test RAG System

In [ ]:
def display_rag_response(response: Dict):
    """Display RAG response in readable format."""
    print(f"\n{'='*80}")
    print(f"Question: {response['query']}")
    print(f"{'='*80}")
    print(f"\nAnswer:\n{response['answer']}")
    print(f"\nConfidence: {response['confidence']:.3f}")
    print(f"\nSources:")
    for i, source in enumerate(response['sources'], 1):
        print(f"  [{i}] {source['title']}")
        print(f"      Section: {source['section']}")
        print(f"      Evidence Level: {source['evidence_level']}")
        print(f"      Relevance: {source['relevance_score']:.3f}")
    print("=" * 80)

# Test clinical queries
clinical_queries = [
    "What is the first-line medication for type 2 diabetes and what is the starting dose?",
    "What are the contraindications for thrombolysis in STEMI patients?",
    "How should I treat a patient with CURB-65 score of 3?",
    "What are the indications for dialysis in acute kidney injury?"
]

responses = []
for query in clinical_queries:
    response = rag_system.generate_response_simulated(query, top_k=3)
    responses.append(response)
    display_rag_response(response)

# Show retrieval statistics
stats = rag_system.get_retrieval_statistics()
print("\nRetrieval Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## Part 7: Production RAG with Real APIs

### Getting Started: API Access

To use RAG in production, you'll need:
1. **Embedding model** (convert text to vectors)
2. **Vector database** (store and search embeddings)
3. **LLM API** (generate responses)

#### Option 1: OpenRouter (Recommended for Students)

**Registration Steps**:
1. Visit: https://openrouter.ai/
2. Click "Sign Up" (top right)
3. Sign in with Google, GitHub, or email
4. Go to "Keys" tab → "Create Key"
5. Copy your API key (starts with `sk-or-v1-...`)
6. **Free credits**: New accounts receive $1-5 in credits (~200-1000 requests)

**Cost Example**:
- Claude 3.5 Haiku: ~$0.25 per 1M input tokens, ~$1.25 per 1M output tokens
- Your $1 credit ≈ 2,700 clinical queries (assuming 500 input + 200 output tokens per query)

#### Option 2: HuggingFace Inference API

**Registration Steps**:
1. Visit: https://huggingface.co/
2. Click "Sign Up" → Create account
3. Go to Settings → Access Tokens
4. Click "New token" → Create token with "read" permissions
5. Copy your token (starts with `hf_...`)
6. **Free tier**: Limited requests per hour, but sufficient for learning

#### Option 3: Google Vertex AI

**Registration Steps**:
1. Visit: https://cloud.google.com/vertex-ai
2. Sign up for Google Cloud (requires credit card, but $300 free credits for 90 days)
3. Enable Vertex AI API
4. Install gcloud CLI and authenticate
5. **Free credits**: $300 for 90 days (thousands of queries)

### Using OpenRouter for RAG

In [ ]:
# Production RAG example (requires API key)
production_rag_example = '''
# PRODUCTION RAG IMPLEMENTATION
# This example shows how to implement RAG with real APIs

import os
import openai
from sentence_transformers import SentenceTransformer
import chromadb

# === SETUP ===

# Option 1: OpenRouter (recommended for students)
openai.api_base = "https://openrouter.ai/api/v1"
openai.api_key = os.getenv("OPENROUTER_API_KEY")

# Load embedding model (runs locally, free)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Initialize ChromaDB
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="clinical_guidelines")

# === INDEXING DOCUMENTS ===

def index_documents(chunks: List[Chunk]):
    """Index documents in ChromaDB with embeddings."""
    for chunk in chunks:
        # Generate embedding
        embedding = embedding_model.encode(chunk.content)

        # Add to ChromaDB
        collection.add(
            ids=[chunk.chunk_id],
            embeddings=[embedding.tolist()],
            documents=[chunk.content],
            metadatas=[chunk.metadata]
        )
    print(f"Indexed {len(chunks)} chunks")

# === RETRIEVAL ===

def retrieve_relevant_chunks(query: str, top_k: int = 3):
    """Retrieve most relevant chunks for query."""
    # Generate query embedding
    query_embedding = embedding_model.encode(query)

    # Search ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    return results['documents'][0], results['metadatas'][0]

# === RAG GENERATION ===

def generate_clinical_answer(query: str):
    """Generate answer using RAG pipeline."""
    # Step 1: Retrieve relevant chunks
    documents, metadatas = retrieve_relevant_chunks(query, top_k=3)

    # Step 2: Construct context
    context = "\\n\\n".join([
        f"[Source {i+1}: {meta['doc_title']}]\\n{doc}"
        for i, (doc, meta) in enumerate(zip(documents, metadatas))
    ])

    # Step 3: Call LLM with context
    response = openai.ChatCompletion.create(
        model="anthropic/claude-3.5-haiku",  # Fast and cost-effective
        messages=[
            {"role": "system", "content": "You are a clinical decision support assistant. Answer based ONLY on provided guidelines. Always cite sources."},
            {"role": "user", "content": f"Clinical Guidelines:\\n{context}\\n\\nQuestion: {query}\\n\\nProvide answer with citations:"}
        ],
        temperature=0.1,  # Low temperature for factual accuracy
        max_tokens=500
    )

    return response.choices[0].message.content

# === USAGE ===

# Index your documents once
index_documents(all_chunks)

# Query the system
answer = generate_clinical_answer(
    "What is the first-line treatment for type 2 diabetes?"
)
print(answer)

# === COST ESTIMATION ===
# Embedding: Free (local model)
# Claude 3.5 Haiku via OpenRouter:
#   - Input: ~$0.25 per 1M tokens
#   - Output: ~$1.25 per 1M tokens
# Example query (500 input + 200 output tokens): ~$0.00037
# Your $1 credit ≈ 2,700 clinical queries
'''

print("Production RAG Implementation Example:")
print(production_rag_example)

## Part 8: Evaluating RAG Performance

### Retrieval Metrics

In [ ]:
class RAGEvaluator:
    """
    Evaluate RAG system performance.
    """

    def __init__(self):
        self.results = []

    def evaluate_retrieval(self,
                          query: str,
                          retrieved_results: List[RetrievalResult],
                          relevant_doc_ids: List[str]) -> Dict:
        """
        Evaluate retrieval quality for a query.

        Args:
            query: The user query
            retrieved_results: Results returned by system
            relevant_doc_ids: Ground truth relevant document IDs

        Returns:
            Metrics: precision, recall, MRR, NDCG
        """
        # Extract retrieved doc IDs
        retrieved_doc_ids = [r.chunk.doc_id for r in retrieved_results]

        # Precision@k: proportion of retrieved docs that are relevant
        relevant_retrieved = set(retrieved_doc_ids) & set(relevant_doc_ids)
        precision = len(relevant_retrieved) / len(retrieved_doc_ids) if retrieved_doc_ids else 0.0

        # Recall@k: proportion of relevant docs that were retrieved
        recall = len(relevant_retrieved) / len(relevant_doc_ids) if relevant_doc_ids else 0.0

        # Mean Reciprocal Rank: 1/rank of first relevant doc
        mrr = 0.0
        for i, doc_id in enumerate(retrieved_doc_ids, 1):
            if doc_id in relevant_doc_ids:
                mrr = 1.0 / i
                break

        # F1 score
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        metrics = {
            'query': query,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'mrr': mrr,
            'num_retrieved': len(retrieved_doc_ids),
            'num_relevant': len(relevant_doc_ids)
        }

        self.results.append(metrics)
        return metrics

    def get_aggregate_metrics(self) -> Dict:
        """Calculate aggregate metrics across all queries."""
        if not self.results:
            return {}

        return {
            'avg_precision': np.mean([r['precision'] for r in self.results]),
            'avg_recall': np.mean([r['recall'] for r in self.results]),
            'avg_f1': np.mean([r['f1'] for r in self.results]),
            'mean_mrr': np.mean([r['mrr'] for r in self.results]),
            'total_queries': len(self.results)
        }

    def visualize_results(self):
        """Visualize evaluation results."""
        if not self.results:
            print("No results to visualize")
            return

        df = pd.DataFrame(self.results)

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Precision per query
        axes[0, 0].bar(range(len(df)), df['precision'], color='steelblue')
        axes[0, 0].set_xlabel('Query Index')
        axes[0, 0].set_ylabel('Precision@k')
        axes[0, 0].set_title('Precision per Query')
        axes[0, 0].axhline(df['precision'].mean(), color='red', linestyle='--', label='Mean')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)

        # Recall per query
        axes[0, 1].bar(range(len(df)), df['recall'], color='coral')
        axes[0, 1].set_xlabel('Query Index')
        axes[0, 1].set_ylabel('Recall@k')
        axes[0, 1].set_title('Recall per Query')
        axes[0, 1].axhline(df['recall'].mean(), color='red', linestyle='--', label='Mean')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)

        # F1 scores
        axes[1, 0].bar(range(len(df)), df['f1'], color='mediumseagreen')
        axes[1, 0].set_xlabel('Query Index')
        axes[1, 0].set_ylabel('F1 Score')
        axes[1, 0].set_title('F1 Score per Query')
        axes[1, 0].axhline(df['f1'].mean(), color='red', linestyle='--', label='Mean')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)

        # MRR scores
        axes[1, 1].bar(range(len(df)), df['mrr'], color='orchid')
        axes[1, 1].set_xlabel('Query Index')
        axes[1, 1].set_ylabel('MRR')
        axes[1, 1].set_title('Mean Reciprocal Rank per Query')
        axes[1, 1].axhline(df['mrr'].mean(), color='red', linestyle='--', label='Mean')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

# Create evaluator
evaluator = RAGEvaluator()

# Create test set with ground truth
test_cases = [
    {
        'query': "What is the first-line treatment for type 2 diabetes?",
        'relevant_docs': ['GUIDE_002']  # Diabetes guideline
    },
    {
        'query': "When should I give thrombolysis for STEMI?",
        'relevant_docs': ['GUIDE_001']  # MI guideline
    },
    {
        'query': "What antibiotics for community-acquired pneumonia?",
        'relevant_docs': ['GUIDE_003']  # Pneumonia guideline
    },
    {
        'query': "How do I manage hypertension in elderly patients?",
        'relevant_docs': ['GUIDE_004']  # Hypertension guideline
    },
    {
        'query': "What are indications for dialysis?",
        'relevant_docs': ['GUIDE_005']  # AKI guideline
    },
    {
        'query': "How to diagnose acute kidney injury?",
        'relevant_docs': ['GUIDE_005']  # AKI guideline
    }
]

# Evaluate retrieval
print("Evaluating RAG retrieval quality...\n")
for test_case in test_cases:
    results = vector_db.search(test_case['query'], top_k=3)
    metrics = evaluator.evaluate_retrieval(
        test_case['query'],
        results,
        test_case['relevant_docs']
    )
    print(f"Query: {test_case['query'][:60]}...")
    print(f"  Precision: {metrics['precision']:.3f}")
    print(f"  Recall: {metrics['recall']:.3f}")
    print(f"  MRR: {metrics['mrr']:.3f}\n")

# Show aggregate metrics
agg_metrics = evaluator.get_aggregate_metrics()
print("\nAggregate Metrics:")
for key, value in agg_metrics.items():
    print(f"  {key}: {value:.3f}")

# Visualize
evaluator.visualize_results()

## Part 9: Real-World Considerations

### Clinical Validation and Safety

In [ ]:
class ClinicalRAGValidator:
    """
    Validates RAG responses for clinical safety.
    """

    def __init__(self, confidence_threshold: float = 0.3):
        self.confidence_threshold = confidence_threshold
        self.validation_log = []

    def validate_response(self, response: Dict) -> Dict:
        """
        Validate RAG response for clinical safety.

        Checks:
        1. Retrieval confidence (are sources relevant?)
        2. Citation presence (is answer grounded?)
        3. Evidence level (what quality of evidence?)
        4. Uncertainty acknowledgment
        """
        validation = {
            'query': response['query'],
            'is_safe': True,
            'warnings': [],
            'recommendations': []
        }

        # Check 1: Low retrieval confidence
        if response['confidence'] < self.confidence_threshold:
            validation['is_safe'] = False
            validation['warnings'].append(
                f"Low retrieval confidence ({response['confidence']:.3f} < {self.confidence_threshold}). "
                "Retrieved documents may not be relevant."
            )
            validation['recommendations'].append(
                "Display warning to user: 'Limited relevant information found. "
                "Please consult additional sources or a specialist.'"
            )

        # Check 2: Missing citations
        if '[Source' not in response['answer']:
            validation['warnings'].append("Answer lacks source citations")
            validation['recommendations'].append(
                "Require LLM to cite sources in response format"
            )

        # Check 3: Evidence level
        evidence_levels = [s.get('evidence_level', 'N/A') for s in response['sources']]
        if all(level == 'N/A' or level in ['C', 'D'] for level in evidence_levels):
            validation['warnings'].append("Low-quality evidence (Level C/D)")
            validation['recommendations'].append(
                "Display evidence level warning to clinician"
            )

        # Check 4: Short answer (possible incomplete response)
        if len(response['answer']) < 50:
            validation['warnings'].append("Answer too brief - may be incomplete")
            validation['recommendations'].append(
                "Review and potentially reject response"
            )

        self.validation_log.append(validation)
        return validation

    def get_safety_report(self) -> Dict:
        """Generate safety report across all validations."""
        if not self.validation_log:
            return {}

        total = len(self.validation_log)
        safe = sum(1 for v in self.validation_log if v['is_safe'])

        return {
            'total_responses': total,
            'safe_responses': safe,
            'unsafe_responses': total - safe,
            'safety_rate': safe / total if total > 0 else 0.0,
            'common_warnings': self._get_common_warnings()
        }

    def _get_common_warnings(self) -> Dict[str, int]:
        """Count occurrence of each warning type."""
        warning_counts = defaultdict(int)
        for validation in self.validation_log:
            for warning in validation['warnings']:
                # Extract warning type (first sentence)
                warning_type = warning.split('.')[0]
                warning_counts[warning_type] += 1
        return dict(warning_counts)

# Test validation
validator = ClinicalRAGValidator(confidence_threshold=0.3)

print("Validating RAG responses...\n")
for response in responses:
    validation = validator.validate_response(response)

    print(f"Query: {validation['query'][:60]}...")
    print(f"  Safe: {validation['is_safe']}")
    if validation['warnings']:
        print(f"  Warnings: {len(validation['warnings'])}")
        for warning in validation['warnings']:
            print(f"    - {warning}")
    print()

# Safety report
safety_report = validator.get_safety_report()
print("\nSafety Report:")
for key, value in safety_report.items():
    if key != 'common_warnings':
        print(f"  {key}: {value}")

if safety_report.get('common_warnings'):
    print("\n  Common warnings:")
    for warning, count in safety_report['common_warnings'].items():
        print(f"    - {warning}: {count}")

## Key Takeaways

### RAG Architecture
1. **RAG = Retrieval + Generation**: Combines search (finding relevant docs) with LLM generation
2. **Reduces hallucinations by 40-60%**: Grounds LLM responses in authoritative sources
3. **Essential for clinical AI**: Guidelines change, LLMs have knowledge cutoffs

### Implementation Components
1. **Document Chunking**: Section-based with overlap, max 512 tokens per chunk
2. **Embeddings**: Use medical domain embeddings (BioBERT, ClinicalBERT) in production
3. **Vector Database**: Pinecone, Weaviate, or ChromaDB for efficient similarity search
4. **Retrieval**: Top-k chunks based on semantic similarity to query
5. **Prompt Construction**: Combine retrieved context + query + instructions
6. **Generation**: LLM generates response with citations

### Evaluation Metrics
- **Precision@k**: Proportion of retrieved docs that are relevant
- **Recall@k**: Proportion of relevant docs that were retrieved
- **MRR**: Mean Reciprocal Rank of first relevant result
- Target: Precision >0.8, Recall >0.7, MRR >0.9 for clinical applications

### Production Considerations
1. **Safety First**: Confidence thresholds, citation requirements, human review
2. **Cost Management**: Embeddings ~$0.02/1M tokens, generation ~$0.25-$2.50/1M tokens
3. **Latency**: Target <3 seconds end-to-end
4. **Monitoring**: Track retrieval quality, LLM costs, user feedback
5. **Compliance**: HIPAA, audit logs, access controls

### Common Pitfalls
- Chunk size too large (low precision) or too small (loses context)
- Generic embeddings instead of medical domain-specific
- No citation tracking (can't verify claims)
- High temperature setting (increases hallucinations)
- No confidence thresholds (dangerous low-quality responses)

### Next Steps
- Test on domain-specific evaluation sets (medical Q&A datasets)
- Implement proper vector database (not TF-IDF)
- Use production embeddings (BioBERT, OpenAI)
- Add reranking with cross-encoder
- Integrate with EHR for patient-specific context

## Exercises

### Exercise 1: Improve Chunking Strategy
Modify `DocumentChunker` to implement a sliding window approach with configurable overlap. Compare retrieval quality with and without overlap.

### Exercise 2: Implement BM25 Ranking
Implement BM25 (Best Matching 25) algorithm as an alternative to TF-IDF for sparse retrieval. Compare performance.

### Exercise 3: Build Query Classifier
Create a classifier that categorizes queries by medical specialty (cardiology, endocrinology, etc.) and filters retrieval to relevant specialties.

### Exercise 4: Citation Verification
Implement a system that verifies each claim in the LLM response can be found in the cited source chunks.

### Exercise 5: Production RAG
Build a complete production RAG system using:
- Sentence-Transformers for embeddings (local, free)
- ChromaDB for vector storage
- OpenRouter with Claude 3.5 Haiku
- Test on 20 clinical queries and measure latency, cost, accuracy

### Exercise 6: Multi-hop RAG
Implement multi-hop reasoning: retrieve documents → generate sub-questions → retrieve again → final answer. Useful for complex clinical scenarios.

### Bonus Challenge: RAG with Patient Context
Extend the system to accept patient context (age, conditions, medications) and filter/rank guidelines based on patient-specific contraindications.